In [22]:
import praw
from pymongo import MongoClient
from transformers import pipeline
import logging
from datetime import datetime

In [23]:
REDDIT_CLIENT_ID = "FPi02ocg4HRZCOdu_CH3Xg"
REDDIT_CLIENT_SECRET = "xVncQKK1nhzCERw-GFBhkmWKeEY_9A"
REDDIT_USER_AGENT = "NewsSummarizer"

In [24]:
REDDIT_CLIENT_ID = "FPi02ocg4HRZCOdu_CH3Xg"
REDDIT_CLIENT_SECRET = "xVncQKK1nhzCERw-GFBhkmWKeEY_9A"
REDDIT_USER_AGENT = "RedditNewsSummarizer"
REDDIT_USERNAME = "Shady-General-6233"
REDDIT_PASSWORD = "Edwin282869"

In [25]:
# Subreddit and Fetch Limit
SUBREDDIT_NAME = "news"  # Example: "worldnews", "technology", "sports"
POST_LIMIT = 5          # Number of top posts to fetch
COMMENT_LIMIT_PER_POST = 10 # Number of top comments to fetch per post


In [26]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Initialize Models ---
try:
    logging.info("Loading summarization model...")
    summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
    logging.info("Summarization model loaded.")

    logging.info("Loading sentiment analysis model...")
    sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    logging.info("Sentiment analysis model loaded.")
except Exception as e:
    logging.error(f"Error loading models: {e}. Please ensure you have an active internet connection or download models locally.")
    exit(1)

2025-07-03 11:01:59,798 - INFO - Loading summarization model...
Device set to use cpu
2025-07-03 11:02:05,302 - INFO - Summarization model loaded.
2025-07-03 11:02:05,303 - INFO - Loading sentiment analysis model...
Device set to use cpu
2025-07-03 11:02:06,435 - INFO - Sentiment analysis model loaded.


In [27]:
# --- Initialize PRAW ---
try:
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT,
        username=REDDIT_USERNAME,
        password=REDDIT_PASSWORD
    )
    logging.info("PRAW initialized successfully.")
except Exception as e:
    logging.error(f"Error initializing PRAW: {e}. Check your Reddit API credentials.")
    exit(1)

2025-07-03 11:02:06,447 - INFO - PRAW initialized successfully.


In [28]:
# --- Initialize MongoDB ---
try:
    client = MongoClient("mongodb://localhost:27017/")
    db = client["redigo_base"]
    collection = db["redigo1"]
    logging.info(f"Connected to MongoDB: Database '{db.name}', Collection '{collection.name}'.")
except Exception as e:
    logging.error(f"Error connecting to MongoDB: {e}")
    exit(1)

2025-07-03 11:02:06,464 - INFO - Connected to MongoDB: Database 'redigo_base', Collection 'redigo1'.


In [29]:
def summarize_text(text: str, default_max_length: int = 150, default_min_length: int = 50) -> str:
   
    if not text:
        return ""

    text_words = text.strip().split()
    text_len_words = len(text_words)

    # If the text is very short, return it as is.
    # This prevents the summarizer from being called with extremely short inputs.
    if text_len_words < default_min_length // 2:
        return text.strip()

    try:
        # Truncate input text if it's excessively long for the model's context window.
        # Most models have a max input length (e.g., 512 or 1024 tokens).
        # Using a character limit as a rough estimate.
        if len(text) > 10000: # Arbitrary large number to prevent extremely long inputs
            text = text[:10000]

        # Dynamically calculate max_length and min_length for the summarizer
        # max_length should be a fraction of the input length, but not exceed default_max_length
        # min_length should also be a fraction, but have a reasonable floor.
        dynamic_max_length = min(max(int(text_len_words * 0.75), default_min_length), default_max_length)
        dynamic_min_length = min(max(int(text_len_words * 0.25), 10), dynamic_max_length - 5) # Ensure min_length is reasonable and less than max

        # Ensure min_length is strictly less than max_length for the summarizer
        if dynamic_min_length >= dynamic_max_length:
            dynamic_min_length = max(10, dynamic_max_length - 10) # Fallback to ensure difference, with a minimum of 10

        summary = summarizer(text, max_length=dynamic_max_length, min_length=dynamic_min_length, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        logging.warning(f"Error summarizing text: {e}. Returning original text for now. Text snippet: '{text[:100]}...'")
        return text

In [ ]:
def analyze_comment_agreement(comments: list, post_title: str, post_selftext: str) -> dict:
    """
    Analyzes sentiment of comments and estimates agreement with the post.
    Agreement is a heuristic based on overall positive/negative sentiment of comments.
    It's a simplification and doesn't involve deep semantic understanding of "agreement".
    """
    total_comments = len(comments)
    if total_comments == 0:
        return {
            "summary": "No comments to analyze.",
            "agreement_percentage": 0,
            "disagreement_percentage": 0,
            "neutral_percentage": 0,
            "comment_details": []
        }

    positive_count = 0
    negative_count = 0
    neutral_count = 0
    comment_summaries = []
    
    # Concatenate post title and selftext for context in sentiment analysis
    post_context = f"{post_title}. {post_selftext}" if post_selftext else post_title

    for comment in comments:
        comment_text = comment.body
        if not comment_text:
            continue
        
        # Summarize comment for brevity
        # FIXED: Changed 'max_length' and 'min_length' to 'default_max_length' and 'default_min_length'
        comment_summary = summarize_text(comment_text, default_max_length=50, default_min_length=10)

        sentiment_score = None
        sentiment_label = "NEUTRAL" # Default to neutral

        try:
            # Analyze sentiment of the comment
            # The sentiment model gives 'POSITIVE', 'NEGATIVE', 'NEUTRAL' labels.
            # We're using the comment's sentiment relative to the general topic.
            sentiment_result = sentiment_analyzer(comment_text[:512]) # Truncate comment for sentiment model if too long
            sentiment_label = sentiment_result[0]['label']
            sentiment_score = sentiment_result[0]['score']

            if sentiment_label == "POSITIVE":
                positive_count += 1
            elif sentiment_label == "NEGATIVE":
                negative_count += 1
            else:
                neutral_count += 1
        except Exception as e:
            logging.warning(f"Error analyzing sentiment for comment: {e}. Comment snippet: '{comment_text[:50]}...'")
            neutral_count += 1 

        comment_summaries.append({
            "id": comment.id,
            "author": str(comment.author),
            "score": comment.score,
            "original_text_snippet": comment_text[:200] + "..." if len(comment_text) > 200 else comment_text,
            "summary": comment_summary,
            "sentiment_label": sentiment_label,
            "sentiment_score": sentiment_score
        })

    agreement_percentage = (positive_count / total_comments) * 100 if total_comments > 0 else 0
    disagreement_percentage = (negative_count / total_comments) * 100 if total_comments > 0 else 0
    neutral_percentage = (neutral_count / total_comments) * 100 if total_comments > 0 else 0

    overall_comment_summary = f"Out of {total_comments} comments: {positive_count} positive, {negative_count} negative, {neutral_count} neutral. "

    if comment_summaries:
        top_comment_snippets = " ".join([c["summary"] for c in comment_summaries if c["summary"]][:5]) # Summarize up to 5 comment summaries
        overall_comment_summary += summarize_text(f"Key themes from comments: {top_comment_snippets}", default_max_length=100, default_min_length=20)
    else:
        overall_comment_summary += "No specific themes identified due to lack of comments."


    return {
        "summary": overall_comment_summary,
        "agreement_percentage": round(agreement_percentage, 2),
        "disagreement_percentage": round(disagreement_percentage, 2),
        "neutral_percentage": round(neutral_percentage, 2),
        "comment_details": comment_summaries
    }


In [ ]:
def fetch_and_process_subreddit(subreddit_name: str, post_limit: int, comment_limit_per_post: int):
    """
    Fetches posts, processes them, and stores the data.
    """
    logging.info(f"Fetching top {post_limit} posts from r/{subreddit_name}...")
    subreddit = reddit.subreddit(subreddit_name)
    processed_posts_count = 0

    for submission in subreddit.top(limit=post_limit): # Or subreddit.new(limit=post_limit) for recent posts
        try:
            logging.info(f"Processing post: '{submission.title}' (ID: {submission.id})")

            # --- NEW FEATURE: Check if post already exists in the database ---
            if collection.find_one({"post_id": submission.id}):
                logging.info(f"Post '{submission.title}' (ID: {submission.id}) already exists in the database. Skipping.")
                continue # Skip to the next submission
            # --- END NEW FEATURE ---

            # Summarize the post title and selftext
            post_text_to_summarize = f"{submission.title}. {submission.selftext}" if submission.selftext else submission.title
            post_summary = summarize_text(post_text_to_summarize)

            # Fetch comments
            submission.comments.replace_more(limit=0) # Flatten comments, get all top-level comments
            comments = [comment for comment in submission.comments.list() if isinstance(comment, praw.models.Comment)][:comment_limit_per_post]
            logging.info(f"Fetched {len(comments)} comments for post '{submission.title}'.")

            # Analyze comments for summary and agreement
            comment_analysis_result = analyze_comment_agreement(comments, submission.title, submission.selftext)

            # Prepare data for MongoDB
            post_data = {
                "post_id": submission.id,
                "title": submission.title,
                "author": str(submission.author),
                "subreddit": submission.subreddit.display_name,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "url": submission.url,
                "created_utc": datetime.fromtimestamp(submission.created_utc),
                "post_summary": post_summary,
                "comments_summary": comment_analysis_result["summary"],
                "comment_agreement_percentage": comment_analysis_result["agreement_percentage"],
                "comment_disagreement_percentage": comment_analysis_result["disagreement_percentage"],
                "comment_neutral_percentage": comment_analysis_result["neutral_percentage"],
                "comment_details": comment_analysis_result["comment_details"],
                "processed_at": datetime.now()
            }

            # Store in MongoDB
            collection.insert_one(post_data)
            logging.info(f"Successfully stored post '{submission.title}' in MongoDB.")
            processed_posts_count += 1

        except Exception as e:
            logging.error(f"Error processing post ID {submission.id} ('{submission.title}'): {e}", exc_info=True)
            continue # Continue to the next post even if one fails

    logging.info(f"Finished processing. Total {processed_posts_count} posts processed and stored.")
    client.close()
    logging.info("MongoDB connection closed.")


In [33]:

if __name__ == "__main__":
    if REDDIT_CLIENT_ID == "YOUR_REDDIT_CLIENT_ID":
        logging.error("Please update REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USERNAME, REDDIT_PASSWORD, and REDDIT_USER_AGENT in the script before running.")
    else:
        fetch_and_process_subreddit(SUBREDDIT_NAME, POST_LIMIT, COMMENT_LIMIT_PER_POST)

2025-07-03 11:02:06,547 - INFO - Fetching newest 5 posts from r/news...
2025-07-03 11:02:08,288 - INFO - Processing post: 'US contractors say their colleagues are firing live ammo as Palestinians seek food in Gaza' (ID: 1lqgbiw)
2025-07-03 11:02:08,739 - INFO - Fetched 10 comments for post 'US contractors say their colleagues are firing live ammo as Palestinians seek food in Gaza'.
2025-07-03 11:02:38,776 - INFO - Successfully stored post 'US contractors say their colleagues are firing live ammo as Palestinians seek food in Gaza' in MongoDB.
2025-07-03 11:02:38,776 - INFO - Processing post: 'Kilmar Abrego Garcia says he was beaten and subjected to psychological torture in El Salvador jail' (ID: 1lqdcha)
2025-07-03 11:02:41,727 - INFO - Fetched 10 comments for post 'Kilmar Abrego Garcia says he was beaten and subjected to psychological torture in El Salvador jail'.
2025-07-03 11:03:04,690 - INFO - Successfully stored post 'Kilmar Abrego Garcia says he was beaten and subjected to psychol